# FL-Fair-Selection sweep on Colab

Runtime -> Change runtime type -> **T4 GPU**, then Run all.
Code is read from `MyDrive/fl-fair-results/` (your uploaded repo folder) and copied to `/content`;
results are written to `MyDrive/fl-fair-results/results/`. Finished runs are skipped, so after a
disconnect just Run all again.

In [1]:
# ---- edit the grid here ----
SELECTORS = ["random", "guarantee", "power_of_choice"]   # add "divfl" if you want it (slow round 0)
ALPHAS    = [0.1, 0.5, 1.0]
SEEDS     = [0, 1, 2]
GOOD      = ["count"]          # guarantee ignores good_def for now, so weighted adds nothing
JOBS      = 2                  # parallel runs on the one GPU; the CNN is tiny
EXTRA     = ""                 # e.g. "--rounds 5" for a quick test

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
SRC = '/content/drive/MyDrive/fl-fair-results'
CODE = '/content/FL-Fair-Selection'
RESULTS = f'{SRC}/results'
os.makedirs(RESULTS, exist_ok=True)
shutil.rmtree(CODE, ignore_errors=True)
shutil.copytree(SRC, CODE, ignore=shutil.ignore_patterns('.venv', '__pycache__', 'results', '.git'))
print(sorted(os.listdir(CODE)))

Mounted at /content/drive
['.gitignore', '.pytest_cache', 'README.md', 'baselines', 'configs', 'conftest.py', 'data', 'fl', 'metrics', 'pytest.ini', 'requirements.txt', 'selection', 'sweep.py', 'tests']


In [3]:
# PR #7 files (main.py, baselines, config, Guarantee registration), not in the uploaded folder
PR7 = {
 "main.py": "\n\"\"\"One config -> one results JSON.\n\nUsage:\n    python main.py                                   # uses configs/default.yaml\n    python main.py --selector power_of_choice --alpha 0.5 --seed 1\n    python main.py --rounds 5                        # quick smoke test\n\nLater, move the loop into fl/train.py and keep this as a thin wrapper.\n\"\"\"\nimport argparse, copy, json, os, random\nimport numpy as np\nimport torch, torch.nn as nn, torch.nn.functional as F\nimport yaml\nfrom torchvision import datasets\n\nfrom baselines import REGISTRY\n\nDEV = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n\n\n# ---------------- data (stub; swap for Person A's get_partition) ----------------\ndef load_partition(cfg):\n    \"\"\"Returns train, test (lists of (X, y) per client). Replace with data.partition\n    once Person A's version is merged, adapting its index output to tensors.\"\"\"\n    alpha, n, seed = cfg[\"alpha\"], cfg[\"n_clients\"], cfg[\"seed\"]\n    rng = np.random.default_rng(seed)\n    ds = datasets.CIFAR10(\"data\", train=True, download=True)\n    X = (torch.tensor(ds.data).permute(0, 3, 1, 2).float() / 255 - 0.5) / 0.5\n    y = np.array(ds.targets)\n    for _ in range(100):                       # resample until every client has >= 10 samples\n        idx = [[] for _ in range(n)]\n        for c in range(10):\n            ids = rng.permutation(np.where(y == c)[0])\n            p = rng.dirichlet([alpha] * n)\n            cuts = (np.cumsum(p) * len(ids)).astype(int)[:-1]\n            for i, part in enumerate(np.split(ids, cuts)):\n                idx[i] += part.tolist()\n        if min(len(i) for i in idx) >= 10:\n            break\n    else:\n        raise RuntimeError(\"could not build a partition with min client size 10\")\n    yt = torch.tensor(y)\n    train, test = [], []\n    for i in range(n):\n        ids = rng.permutation(idx[i]); s = int(0.8 * len(ids))\n        train.append((X[ids[:s]], yt[ids[:s]])); test.append((X[ids[s:]], yt[ids[s:]]))\n    return train, test\n\n\n# ---------------- model + helpers ----------------\nclass CNN(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.c1, self.c2 = nn.Conv2d(3, 32, 3), nn.Conv2d(32, 64, 3)\n        self.f1, self.f2 = nn.Linear(64 * 6 * 6, 128), nn.Linear(128, 10)\n\n    def forward(self, x):\n        x = F.max_pool2d(F.relu(self.c1(x)), 2)\n        x = F.max_pool2d(F.relu(self.c2(x)), 2)\n        return self.f2(F.relu(self.f1(x.flatten(1))))\n\n\ndef train_local(model, X, y, cfg):\n    model.to(DEV).train()\n    opt = torch.optim.SGD(model.parameters(), lr=cfg[\"lr\"], momentum=0.9)\n    bs = cfg[\"batch_size\"]\n    for _ in range(cfg[\"local_epochs\"]):\n        perm = torch.randperm(len(X))\n        for i in range(0, len(X), bs):\n            b = perm[i:i + bs]\n            opt.zero_grad()\n            F.cross_entropy(model(X[b].to(DEV)), y[b].to(DEV)).backward()\n            opt.step()\n\n\n@torch.no_grad()\ndef evaluate(model, X, y):\n    \"\"\"returns (mean loss, accuracy)\"\"\"\n    if len(X) == 0:\n        return 0.0, 0.0\n    model.to(DEV).eval()\n    loss = correct = 0\n    for i in range(0, len(X), 512):\n        xb, yb = X[i:i + 512].to(DEV), y[i:i + 512].to(DEV)\n        out = model(xb)\n        loss += F.cross_entropy(out, yb, reduction=\"sum\").item()\n        correct += (out.argmax(1) == yb).sum().item()\n    return loss / len(X), correct / len(X)\n\n\ndef flat(m):\n    return torch.cat([p.detach().flatten() for p in m.parameters()]).cpu()\n\n\nclass Ctx:\n    \"\"\"What a selector may ask the server for. Random ignores it entirely.\"\"\"\n    def __init__(self):\n        self.loss_fn = None      # ids -> losses of current global model (Power-of-Choice)\n        self.update_fn = None    # ids -> local update vectors (DivFL)\n        self.updates = {}        # {client_id: update vector} for clients trained this round\n\n\n# ---------------- main ----------------\ndef run(cfg):\n    random.seed(cfg[\"seed\"]); np.random.seed(cfg[\"seed\"]); torch.manual_seed(cfg[\"seed\"])\n    train, test = load_partition(cfg)\n    n, k = cfg[\"n_clients\"], cfg[\"k\"]\n    sizes = [len(t[0]) for t in train]\n\n    registry = dict(REGISTRY)\n    try:                                         # pick up Rahul's selectors if registered\n        from selection import REGISTRY as TEAM\n        registry.update(TEAM)\n    except Exception:\n        pass\n    if cfg[\"selector\"] not in registry:\n        raise KeyError(f\"unknown selector {cfg['selector']}; have {list(registry)}\")\n    selector = registry[cfg[\"selector\"]](n, k, sizes, cfg)\n\n    glob = CNN().to(DEV)\n    n_test = [len(t[0]) for t in test]\n    counts, sel_log, acc_log, client_log, query_log = [0] * n, [], [], [], []\n\n    def loss_fn(ids):\n        return [evaluate(glob, *train[i])[0] for i in ids]\n\n    def update_fn(ids):\n        out = []\n        for i in ids:\n            m = copy.deepcopy(glob); train_local(m, *train[i], cfg)\n            out.append((flat(m) - flat(glob)).numpy())\n        return out\n\n    for r in range(cfg[\"rounds\"]):\n        ctx = Ctx(); ctx.loss_fn, ctx.update_fn = loss_fn, update_fn\n        chosen = [int(i) for i in selector.select(r, ctx)]\n        assert len(set(chosen)) == k, f\"selector returned {len(chosen)} ids, expected {k} distinct\"\n        states, ws = [], []\n        g0 = flat(glob)\n        for i in chosen:\n            m = copy.deepcopy(glob); train_local(m, *train[i], cfg)\n            ctx.updates[i] = (flat(m) - g0).numpy()\n            states.append({kk: v.detach().clone() for kk, v in m.state_dict().items()})\n            ws.append(sizes[i]); counts[i] += 1\n        tot = sum(ws)                            # data-size-weighted FedAvg over selected clients\n        glob.load_state_dict({kk: sum(s[kk] * (w / tot) for s, w in zip(states, ws))\n                              for kk in states[0]})\n        selector.update(r, chosen, ctx)\n        sel_log.append(chosen)\n        q = int(getattr(selector, \"extra_queries\", 0))\n        query_log.append(q - sum(query_log))\n        # global acc on the union of client test splits = test-size-weighted client acc\n        accs = [evaluate(glob, *test[i])[1] for i in range(n)]\n        client_log.append(accs)\n        acc_log.append(sum(a * m for a, m in zip(accs, n_test)) / sum(n_test))\n        print(f\"round {r:3d}  global_acc {acc_log[-1]:.3f}\", flush=True)\n\n    out = {\n        \"cfg\": {kk: v for kk, v in cfg.items()},\n        \"selected_ids\": sel_log,\n        \"global_acc\": acc_log,                   # on the union of client test splits\n        \"client_acc\": client_log[-1],            # final per-client accuracy\n        \"client_acc_rounds\": client_log,         # per-client accuracy after each round\n        \"selection_counts\": counts,\n        \"data_sizes\": sizes,\n        \"queried_per_round\": query_log,          # extra client queries in each round\n        \"extra_queries\": sum(query_log),\n    }\n    os.makedirs(\"results\", exist_ok=True)\n    path = f\"results/{cfg['selector']}_{cfg['dataset']}_{cfg['alpha']}_{cfg['good_def']}_{cfg['seed']}.json\"\n    json.dump(out, open(path, \"w\"))\n    print(\"wrote\", path)\n\n\nif __name__ == \"__main__\":\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--config\", default=\"configs/default.yaml\")\n    for name, typ in [(\"selector\", str), (\"alpha\", float), (\"seed\", int), (\"rounds\", int),\n                      (\"n_clients\", int), (\"k\", int), (\"lr\", float), (\"d\", int),\n                      (\"good_def\", str), (\"local_epochs\", int)]:\n        ap.add_argument(f\"--{name}\", type=typ, default=None)\n    args = ap.parse_args()\n    cfg = yaml.safe_load(open(args.config))\n    for kk, v in vars(args).items():\n        if kk != \"config\" and v is not None:\n            cfg[kk] = v\n    run(cfg)",
 "baselines/__init__.py": "from .random_sel import RandomSelector\nfrom .power_of_choice import PowerOfChoiceSelector\nfrom .divfl import DivFLSelector\n\nREGISTRY = {\n    \"random\": RandomSelector,\n    \"power_of_choice\": PowerOfChoiceSelector,\n    \"divfl\": DivFLSelector,\n}",
 "baselines/random_sel.py": "# import numpy as np\n# from selection.base import Selector   # adjust to whatever base.py actually exports\n\n\n# class RandomSelector(Selector):\n#     \"\"\"FedAvg-style uniform sampling of K clients without replacement.\"\"\"\n\n#     def __init__(self, n_clients, k, data_sizes, cfg):\n#         self.n = n_clients\n#         self.k = k\n#         seed = cfg.get(\"seed\", 0) if isinstance(cfg, dict) else getattr(cfg, \"seed\", 0)\n#         self.rng = np.random.default_rng(seed)   # own RNG, so runs reproduce per seed\n\n#     def select(self, round_idx, client_losses):\n#         return self.rng.choice(self.n, size=self.k, replace=False).tolist()\n\n#     def update(self, round_idx, selected, client_losses):\n#         pass   # Random keeps no state\n\nimport numpy as np\n\n\nclass RandomSelector:\n    \"\"\"FedAvg-style uniform sampling of K clients without replacement.\"\"\"\n\n    def __init__(self, n_clients, k, data_sizes, cfg):\n        self.n, self.k = n_clients, k\n        self.rng = np.random.default_rng(cfg.get(\"seed\", 0))\n        self.extra_queries = 0          # Random needs no extra client queries\n\n    def select(self, round_idx, ctx):\n        return [int(i) for i in self.rng.choice(self.n, size=self.k, replace=False)]\n\n    def update(self, round_idx, selected, ctx):\n        pass",
 "baselines/power_of_choice.py": "import numpy as np\n\n\nclass PowerOfChoiceSelector:\n    \"\"\"Sample d candidates (prob. proportional to data size), evaluate the current\n    global model on each, keep the top-K by loss.  (Cho, Wang, Joshi 2020)\n\n    VERIFY against the paper: candidate sampling, loss basis (full local data here),\n    and the value of d (default 2*K is a guess).\n    \"\"\"\n\n    def __init__(self, n_clients, k, data_sizes, cfg):\n        self.n, self.k = n_clients, k\n        self.d = int(cfg.get(\"d\") or 2 * k)\n        assert self.k <= self.d <= self.n, \"need K <= d <= N\"\n        sizes = np.asarray(data_sizes, dtype=float)\n        self.p = sizes / sizes.sum()\n        self.rng = np.random.default_rng(cfg.get(\"seed\", 0))\n        self.extra_queries = 0          # loss evaluations on non-selected clients too\n\n    def select(self, round_idx, ctx):\n        cand = self.rng.choice(self.n, size=self.d, replace=False, p=self.p)\n        losses = np.asarray(ctx.loss_fn(cand))      # loss of CURRENT global model\n        self.extra_queries += self.d\n        top = np.argsort(-losses)[: self.k]         # highest loss first\n        return [int(c) for c in cand[top]]\n\n    def update(self, round_idx, selected, ctx):\n        pass",
 "baselines/divfl.py": "# import numpy as np\n# from selection.base import Selector   # match base.py\n\n\n# class DivFLSelector(Selector):\n#     \"\"\"Greedy facility-location selection over client update vectors (Balakrishnan et al., ICLR 2022).\"\"\"\n\n#     def __init__(self, n_clients, k, data_sizes, cfg):\n#         self.n, self.k = n_clients, k\n#         self.G = None                  # (N, P) last known update per client\n#         self.seen = np.zeros(n_clients, dtype=bool)\n#         self.query_count = 0           # extra client computations, for the communication claim\n\n#     def select(self, round_idx, update_fn):\n#         # warm-up: get an update from every client that has none yet (all N in round 0)\n#         missing = np.where(~self.seen)[0]\n#         if len(missing) > 0:\n#             U = np.asarray(update_fn(missing))\n#             if self.G is None:\n#                 self.G = np.zeros((self.n, U.shape[1]), dtype=np.float32)\n#             self.G[missing] = U\n#             self.seen[missing] = True\n#             self.query_count += len(missing)\n#         return self._greedy()\n\n#     def _greedy(self):\n#         G = self.G\n#         sq = (G ** 2).sum(1)\n#         D = np.sqrt(np.maximum(sq[:, None] + sq[None, :] - 2 * G @ G.T, 0))   # pairwise distances\n#         sim = D.max() - D                                                     # similarity >= 0\n#         best = np.zeros(self.n)        # best similarity of each client to the chosen set so far\n#         chosen = []\n#         for _ in range(self.k):\n#             gain = np.maximum(sim, best[:, None]).sum(0) - best.sum()\n#             gain[chosen] = -np.inf\n#             j = int(np.argmax(gain))\n#             chosen.append(j)\n#             best = np.maximum(best, sim[:, j])\n#         return chosen\n\n#     def update(self, round_idx, selected, updates):\n#         # updates: dict {client_id: flat update vector} for the clients just trained\n#         for i in selected:\n#             self.G[i] = updates[i]\n\nimport numpy as np\n\n\nclass DivFLSelector:\n    \"\"\"Greedy facility-location selection over client update vectors\n    (Balakrishnan et al., ICLR 2022).\n\n    Simplification: round 0 gets an update from every client; afterwards each client's\n    stored update is refreshed only when it is selected (stale otherwise).\n    VERIFY against the paper: gradient vs update, staleness handling, and aggregation\n    weights (paper may weight by cluster size; we keep data-size FedAvg on purpose).\n    \"\"\"\n\n    def __init__(self, n_clients, k, data_sizes, cfg):\n        self.n, self.k = n_clients, k\n        self.G = None                               # (N, P) last known update per client\n        self.seen = np.zeros(n_clients, dtype=bool)\n        self.extra_queries = 0\n\n    def select(self, round_idx, ctx):\n        missing = np.where(~self.seen)[0]\n        if len(missing) > 0:\n            U = np.asarray(ctx.update_fn(missing), dtype=np.float32)\n            if self.G is None:\n                self.G = np.zeros((self.n, U.shape[1]), dtype=np.float32)\n            self.G[missing] = U\n            self.seen[missing] = True\n            self.extra_queries += len(missing)\n        return self._greedy()\n\n    def _greedy(self):\n        G = self.G\n        sq = (G ** 2).sum(1)\n        D = np.sqrt(np.maximum(sq[:, None] + sq[None, :] - 2 * G @ G.T, 0))\n        sim = D.max() - D                           # similarity >= 0\n        best = np.zeros(self.n)                     # best similarity to chosen set so far\n        chosen = []\n        for _ in range(self.k):\n            gain = np.maximum(sim, best[:, None]).sum(0) - best.sum()\n            gain[chosen] = -np.inf\n            j = int(np.argmax(gain))\n            chosen.append(j)\n            best = np.maximum(best, sim[:, j])\n        return chosen\n\n    def update(self, round_idx, selected, ctx):\n        for i in selected:\n            self.G[i] = ctx.updates[i]",
 "configs/default.yaml": "dataset: cifar10\nalpha: 0.1\nn_clients: 50\nk: 10\nrounds: 100\nlocal_epochs: 2\nbatch_size: 32   # not set before; main.py needs it\nlr: 0.01        # tune once on Random, then freeze\nseed: 0\nselector: random\ngood_def: count # count | weighted\nd: null               # Power-of-Choice candidate set size (null -> 2*k)\nT: 10\nM: 1\neps: 0.1\nfiller: random",
 "selection/__init__.py": "from .guarantee import GuaranteeSelector\n\nREGISTRY = {\"guarantee\": GuaranteeSelector}\n"
}
for f, txt in PR7.items():
    path = os.path.join(CODE, f)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    open(path, 'w').write(txt)
print('wrote', ', '.join(PR7))

wrote main.py, baselines/__init__.py, baselines/random_sel.py, baselines/power_of_choice.py, baselines/divfl.py, configs/default.yaml, selection/__init__.py


In [4]:
%cd /content/FL-Fair-Selection
!ln -sfn {RESULTS} results
!nvidia-smi -L
!python -c "import torch; print('cuda', torch.cuda.is_available())"

/content/FL-Fair-Selection
GPU 0: Tesla T4 (UUID: GPU-988bed34-c075-acc9-f5aa-d0275a07e981)
cuda True


In [5]:
# timing check: 3 rounds of Random, written outside results/ so the sweep is unaffected
import time, subprocess
t = time.time()
subprocess.run("python main.py --rounds 3 --seed 99 > /tmp/timing.log 2>&1; mv results/random_cifar10_0.1_count_99.json /tmp/ 2>/dev/null", shell=True)
print(f"~{(time.time() - t) / 3:.1f} s per round incl. CIFAR load; a 100-round run is about 100x that")
!tail -2 /tmp/timing.log

~659.6 s per round incl. CIFAR load; a 100-round run is about 100x that
round   2  global_acc 0.189
wrote results/random_cifar10_0.1_count_99.json


In [6]:
cmd = (f"python sweep.py --selectors {' '.join(SELECTORS)} --alphas {' '.join(map(str, ALPHAS))} "
       f"--seeds {' '.join(map(str, SEEDS))} --good {' '.join(GOOD)} --jobs {JOBS}")
if EXTRA:
    cmd += f" -- {EXTRA}"
print(cmd)
!{cmd} --dry-run | head -3
!{cmd}

python sweep.py --selectors random guarantee power_of_choice --alphas 0.1 0.5 1.0 --seeds 0 1 2 --good count --jobs 2
27 to run, 0 already done
  python main.py --config configs/default.yaml --selector random --alpha 0.1 --good_def count --seed 0
  python main.py --config configs/default.yaml --selector random --alpha 0.1 --good_def count --seed 1
27 to run, 0 already done
[1/27] random alpha=0.1 good=count seed=0  220s  ok
[2/27] random alpha=0.1 good=count seed=1  224s  ok
[3/27] random alpha=0.1 good=count seed=2  228s  ok
[4/27] random alpha=0.5 good=count seed=0  232s  ok
[5/27] random alpha=0.5 good=count seed=1  221s  ok
[6/27] random alpha=0.5 good=count seed=2  226s  ok
[7/27] random alpha=1.0 good=count seed=0  218s  ok
[8/27] random alpha=1.0 good=count seed=1  219s  ok
[9/27] random alpha=1.0 good=count seed=2  214s  ok
[10/27] guarantee alpha=0.1 good=count seed=0  217s  ok
[11/27] guarantee alpha=0.1 good=count seed=1  221s  ok
[12/27] guarantee alpha=0.1 good=count seed=

In [7]:
# summary tables + figures, then zip results/ only
!python -m metrics.loader results --out results/summary.csv
!python -m metrics.gate1 results/random_*.json --out results/gate1_pooled.csv | sed -n '/Pooled/,$p'
!python -m metrics.compare results --out results/compare.csv --latex results/tables/main_{alpha}.tex > /dev/null
!python -m metrics.plots results --out results/figures
!cd {SRC} && rm -f results.zip && zip -qr results.zip results -x 'results/logs/*' && ls -lh results.zip

       selector dataset  alpha  good  n_seeds global_acc_final worst10_mean_final        min_final   variance_final       jain_final         cv_final        sel_freq_var queries_per_round   rounds_to_target  n_reached
      guarantee cifar10    0.1 count        3 0.5510 +- 0.0313   0.2414 +- 0.0855 0.1304 +- 0.0607 0.0273 +- 0.0090 0.9227 +- 0.0273 0.2865 +- 0.0584    0.6800 +- 0.0693  0.0000 +- 0.0000  44.3333 +- 9.0185          3
      guarantee cifar10    0.5 count        3 0.6626 +- 0.0091   0.5248 +- 0.0341 0.4823 +- 0.0323 0.0048 +- 0.0013 0.9892 +- 0.0032 0.1037 +- 0.0164    0.6800 +- 0.0693  0.0000 +- 0.0000  36.3333 +- 1.5275          3
      guarantee cifar10    1.0 count        3 0.6714 +- 0.0059   0.5908 +- 0.0035 0.5731 +- 0.0043 0.0023 +- 0.0005 0.9949 +- 0.0010 0.0717 +- 0.0070    0.6800 +- 0.0693  0.0000 +- 0.0000  32.6667 +- 2.0817          3
power_of_choice cifar10    0.1 count        3 0.5323 +- 0.0102   0.2051 +- 0.0755 0.1553 +- 0.0735 0.0271 +- 0.0048 0.9126 +- 0.

Send back `MyDrive/fl-fair-results/results.zip` (or the `results/*.json` files) and the figures go into the PRs.